In [ ]:
# Import the libraries required for data handling, modeling, and evaluation.
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [2]:
# Set a fixed random seed to make model training and validation reproducible.
seed = 356437451

In [3]:
# Load the training and external validation datasets from CSV files.
train = pd.read_csv('data_train.csv')
X_ini = train.iloc[:,2:]
y_ini = train.iloc[:,1]

In [4]:
# Create repeated stratified cross-validation splits for robust validation.
sp = RepeatedStratifiedKFold(random_state=seed,n_repeats=3,n_splits=5)

In [5]:
# Import the libraries required for data handling, modeling, and evaluation.
import hyperopt
from hyperopt import hp

In [6]:
# Define the hyperparameter objective function optimized by cross-validation AUC.
def objective(param):
    aucs = []
    for train_index,test_index in sp.split(X_ini,y_ini):
        X_train = X_ini.iloc[train_index,:]
        X_vali = X_ini.iloc[test_index,:]
        y_train = y_ini[train_index]
        y_vali = y_ini[test_index]
        # Instantiate the classifier using the selected or predefined hyperparameters.
        model = RandomForestClassifier(random_state=seed,
                                       n_estimators=param['n_estimators'],
                                       max_depth=param['max_depth'],
                                       min_samples_split=param['min_samples_split'],
                                       min_samples_leaf=param['min_samples_leaf'])
        # Fit the model on the training cohort.
        model.fit(X_train,y_train)
        # Generate predicted probabilities for ROC-AUC evaluation.
        pro_vali = model.predict_proba(X_vali)[:,1]
        # Report model performance metrics for this cohort.
        auc_vali = roc_auc_score(y_vali,pro_vali)
        aucs.append(auc_vali)
    return -np.mean(aucs)

In [ ]:
# Specify the hyperparameter search space for Bayesian optimization.
space = {
    'n_estimators':hp.choice('n_estimators',range(2,50)),
    'max_depth':hp.choice('max_depth',range(1,3)),
    'min_samples_split':hp.choice('min_samples_split',range(2,50)),
    'min_samples_leaf':hp.choice('min_samples_leaf',range(2,50)),
}

In [8]:
# Run Bayesian hyperparameter optimization and keep the best parameter indices.
best_param = hyperopt.fmin(objective,space,hyperopt.tpe.suggest,max_evals=100)

100%|████████████████████████████████████████████████████████████| 100/100 [00:54<00:00,  1.83trial/s, best loss: -1.0]


In [9]:
# Display the best hyperparameter setting selected during optimization.
best_param

{'max_depth': np.int64(1),
 'min_samples_leaf': np.int64(6),
 'min_samples_split': np.int64(3),
 'n_estimators': np.int64(47)}

In [ ]:
# Instantiate the classifier using the selected or predefined hyperparameters.
model = RandomForestClassifier(random_state=seed,
                               # Display the best hyperparameter setting selected during optimization.
                               n_estimators=range(2,50)[best_param['n_estimators']],
                               max_depth=range(1,3)[best_param['max_depth']],
                               min_samples_split=range(2,50)[best_param['min_samples_split']],
                               min_samples_leaf=range(2,50)[best_param['min_samples_leaf']])
# Fit the model on the training cohort.
model.fit(X_ini,y_ini)

RandomForestClassifier(max_depth=2, min_samples_leaf=8, min_samples_split=5,
                       n_estimators=49, random_state=356437451)

In [ ]:
# Generate predicted probabilities for ROC-AUC evaluation.
pro_train = model.predict_proba(X_ini)[:,1]
# Report model performance metrics for this cohort.
print('train set AUC={:.3f}'.format(roc_auc_score(y_ini,pro_train)))

In [12]:
df_train = pd.DataFrame({
    'ID':train['ID'],
    'True':y_ini,
    'Pre':pro_train
})
# Export predicted labels so downstream confusion-matrix analyses can reuse them.
df_train.to_csv('RF_train.csv',index=False)

In [ ]:
test_files = ['data_test1.csv', 'data_test2.csv', 'data_test3.csv', 'data_test4.csv', 'data_test5.csv', 'data_test6.csv']
for test_file in test_files:
    # Load the training and external validation datasets from CSV files.
    test = pd.read_csv(test_file)
    X_test = test.iloc[:,2:]
    y_test = test.iloc[:,1]

    # Generate predicted probabilities for ROC-AUC evaluation.
    pro_test = model.predict_proba(X_test)[:,1]

    # Report model performance metrics for this cohort.
    print(f'{test_file} Test set AUC={roc_auc_score(y_test, pro_test):.3f}')

    df_test = pd.DataFrame({
        'ID': test['ID'],
        'True': y_test,
        'Pre': pro_test
    })
    # Export predicted labels so downstream confusion-matrix analyses can reuse them.
    df_test.to_csv(f'RF_{test_file}_predictions.csv', index=False)

In [ ]:
# Import the libraries required for data handling, modeling, and evaluation.
import joblib
# Save the trained model for later reuse.
joblib.dump(model, 'saved_model/RF.pkl')